## Modelo de Toomre

1. *Núcelos Puntuales* : representan toda la masa de cada galaxia, es decir, las interacciones gravitacionales que se dan con y entre estos núcleos.

2. *Estrellas como partículas de prueba* : la masa de las estrellas es m = 0 y esto quiere decir que no interactúan entre ellas


In [1]:
!pip install numba


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\ASUS\Downloads\Mecanica_cel\Mec-nica_Celeste-main\mecanicacelev\Scripts\python.exe -m pip install --upgrade pip


In [7]:
import numpy as np
import matplotlib.pyplot as plt 
from numba import jit
from typing import List, Dict, Any

In [19]:
class N_system: 
    def __init__(self, particles: List[Dict[str, Any]]):
        self.n = len(particles)

        self.positions = np.array([p['position'] for p in particles])
        self.velocities = np.array([p['velocity'] for p in particles])
        self.masses = np.array([p['mass'] for p in particles])
        self.mass = self.masses.sum()

        v_prom = np.mean(np.linalg.norm(self.velocities, axis=1)) 
        self.t_relax =  self.n / np.log(self.n) * (v_prom**3) / (self.mass * 1.0)

        ## Índices de cuerpos con masa
        self.mass_indices = np.where(self.masses > 0)[0]    


    def evolucionar(self, dt: float, T: float) -> np.ndarray[float, 3]:

        ''' Integra el sistema por el método de leapgrog durante un tiempo T con paso dt
        
        params;
        - dt: paso de tiempo
        - T: tiempo totalde evolución
        returns:
        - trayectories: array de forma (n, pasos, 3)
          con las posiciones de cada partícula en cada paso
        '''
        
        def acc(pos: np.ndarray[float, 3],
                        masses: np.ndarray[float],
                        m_indices: np.ndarray[int],
                        n_particles: int) -> np.ndarray[float, 3]:
            ''' Calcula la aceleración de cada partícula dada su posición
            
            params:
            - pos: array de forma (n, 3) con las posiciones de cada partícula
            returns:
            - acc: array de forma (n, 3) con las aceleraciones de cada partícula
            '''
            acc = np.zeros_like(n_particles, pos)
            for i in range(self.n):
                for j in m_indices:

                    if i == j:
                        continue

                    r_vec = pos[j] - pos[i] # Vector de posición de j respecto a i
                    r_mag = np.linalg.norm(r_vec) + 1e-10 # Magnitud del vector de posición
                    acc -= self.masses[j] * r_vec / r_mag**3
            return acc
            
            num_steps = int(T / dt)
            trajectories = np.zeros((self.n, num_steps, 3))

            trajectories[:, 0, :] = self.positions # Guardamos la posición inicial

    #Leapfrog integration

            accelerations = acc(self.positions, self.masses, self.mass_indices, self.n)
            self.velocities += 0.5 * accelerations * dt # Paso de velocidad a mitad de paso

            for step in range(1, num_steps):
                self.positions += self.velocities * dt # Paso de posición
                trajectories[:, step, :] = self.positions # Guardamos la posición

                accelerations = acc(self.positions, self.masses, self.mass_indices, self.n) # Calculamos aceleración en la nueva posición
                self.velocities += accelerations * dt # Paso de velocidad completo

            return trajectories

In [20]:
mi_primer_sistema = N_system([
    {'mass' : 1.0, 'position' : [0.0, 0.0, 0.0], 
     'velocity' : [0.0, 0.0, 0.0]},
    {'mass' : 2.0, 'position' : [1.0, 0.0, 0.0],
      'velocity' : [0.0, 1.0, 0.0]},
    {'mass' : 3.0, 'position' : [0.0, 1.0, 0.0],
      'velocity' : [0.0, 0.0, 1.0]}
])

In [21]:
mi_primer_sistema.evolucionar(0.01, 1.0)